# 3-Phase Frequent Itemset Extractor Training (Qwen 2.5-7B) — v3

**Pipeline:** SFT with Chain-of-Thought → DPO with Real LLM Failures *(GRPO skipped in v3)*

## What's different in v3 (LLM Council recommendations)
| Aspect | v2 | v3 (this notebook) |
|--------|----|--------------------|
| LoRA rank | r=64, alpha=16 (ratio 0.25) | r=**32**, alpha=**64** (ratio **2.0**) |
| LoRA dropout | 0 | **0.05** |
| Seq length | 4096 | **4096** *(restored in v3.7 after finding SFT truncation at 2048)* |
| SFT config | 2 epochs, lr=2e-4 | **3** epochs, lr=**1e-4**, eval every 50 steps |
| DPO config | 2 epochs | **1** epoch (overfits quickly) |
| GRPO | Full 200 steps | **SKIPPED** (add in v4 if needed) |
| Save method | `merged_4bit_forced` ❌ | **Adapter-only** via `save_pretrained` ✅ |
| Inference temp | 0.1 | **0.3** + top_k=50, top_p=0.90 |
| CoT format | Verbose with evidence rows | **Concise** column-grouped format |

## Phases
1. **SFT-CoT** — Teach the model to reason step-by-step using `<think>` tags, then output JSON
2. **DPO-Real** — Steer away from real mistakes LLMs actually made (hallucinated evidence rows, wrong counts)
3. **GRPO** *(skipped in v3)* — Can be added later as refinement pass

## Requirements
- GPU with ≥16 GB VRAM (T4, A100, L4, H200, etc.)
- Dataset on HuggingFace Hub (uploaded by `build_hf_dataset_v2.py`)
- ~1-2 hours total training time on H200/A100

In [ ]:
# ── Cell 1: Install dependencies (run ONCE, then restart kernel) ──────────────
# ⚠️ TLJH server has a working torch 2.7.1+cu118 at /opt/tljh/user/.
#    Installing unsloth pulls torch 2.10+ to ~/.local which SHADOWS system torch
#    and crashes (CUDA 11.8 vs CUDA 12.x mismatch).
#
#    Fix: install deps → delete ONLY core torch + nvidia cu12x → restart kernel.
#    We KEEP torchvision/torchaudio (unsloth imports them, not used for training).
#    The cell is IDEMPOTENT: safe to re-run, won't re-pull torch if deps exist.

import os, glob, shutil, subprocess

USER_SP = os.path.expanduser("~/.local/lib/python3.12/site-packages")

# ── Step 1: Only install if unsloth is NOT already present ────────────────────
unsloth_dir = os.path.join(USER_SP, "unsloth")
if not os.path.isdir(unsloth_dir):
    print("📦 First run — installing ML packages...")
    subprocess.check_call(
        "pip install unsloth trl datasets transformers accelerate "
        "bitsandbytes huggingface_hub peft safetensors sentencepiece protobuf -q".split()
    )
    print("📦 Install complete.")
else:
    print("✅ Packages already installed — skipping pip install")

# ── Step 2: Remove ONLY core torch + nvidia (keep torchvision/torchaudio) ────
# We delete: torch, torch-*, nvidia* (cu12x CUDA libs that conflict with cu118)
# We keep:   torchvision, torchaudio, triton (unsloth imports them, harmless)
removed = []
for pattern in ["torch", "torch-*"]:
    for p in glob.glob(os.path.join(USER_SP, pattern)):
        basename = os.path.basename(p)
        # Skip torchvision and torchaudio — unsloth needs them importable
        if basename.startswith("torchvision") or basename.startswith("torchaudio"):
            continue
        shutil.rmtree(p, ignore_errors=True)
        removed.append(basename)

# Remove nvidia cu12x runtime libs (conflict with system CUDA 11.8)
for p in glob.glob(os.path.join(USER_SP, "nvidia*")):
    shutil.rmtree(p, ignore_errors=True)
    removed.append(os.path.basename(p))

if removed:
    print(f"🗑️  Cleaned from ~/.local: {removed}")
else:
    print("✅ No user-level torch/nvidia to clean")

# ── Step 3: Verify system torch is reachable ─────────────────────────────────
assert not os.path.exists(os.path.join(USER_SP, "torch")), \
    f"❌ FAILED: torch still in {USER_SP}/torch — please delete manually"

r = subprocess.run(
    ["python3", "-c", "import torch; print(torch.__version__, torch.__file__)"],
    capture_output=True, text=True
)
if r.returncode == 0:
    print(f"✅ System torch: {r.stdout.strip()}")
else:
    print(f"⚠️  System torch not found: {r.stderr.strip()[:200]}")

print("\n" + "=" * 60)
print("⚠️  RESTART THE KERNEL (Kernel → Restart)")
print("   Then run cell 2 (CONFIG) onwards. Re-running this cell is safe.")
print("=" * 60)

In [ ]:
# ── Cell 2: CONFIG — edit this cell only ─────────────────────────────────────
# v3.11 (2026-03-18): Proper HF versioning — v3 dataset in its own repo.
# v3.10: Fixed R-shorthand tokenization ambiguity, 272 examples with spaced R-refs.
CONFIG = {
    # ── Model ────────────────────────────────────────────────────────────────
    "base_model":        "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    "max_seq_length":    4096,    # v3.7: restored from 2048 → 4096 to avoid truncating SFT CoT+JSON targets
    "load_in_4bit":      True,

    # ── Dataset (HuggingFace Hub) ─────────────────────────────────────────────
    "hf_dataset":        "OliverSlivka/itemset-extraction-v3",
    "hf_token":          "",    # paste HF token here, or set env HF_TOKEN

    # ── LoRA ─────────────────────────────────────────────────────────────────
    "lora_r":            32,
    "lora_alpha":        64,
    "lora_dropout":      0.05,
    "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                             "gate_proj", "up_proj", "down_proj"],

    # ── Phase 1: SFT with CoT ────────────────────────────────────────────────
    "sft_epochs":        3,
    "sft_lr":            1e-4,
    "sft_batch_size":    2,      # reduce to 1 if OOM
    "sft_grad_accum":    4,
    "sft_warmup_ratio":  0.10,
    "sft_weight_decay":  0.01,
    "sft_output_dir":    "./sft_checkpoint",

    # ── Phase 2: DPO with Real Failures ──────────────────────────────────────
    "dpo_epochs":        1,
    "dpo_lr":            5e-5,
    "dpo_beta":          0.1,
    "dpo_batch_size":    1,
    "dpo_grad_accum":    4,
    "dpo_output_dir":    "./dpo_checkpoint",

    # ── Phase 3: GRPO — SKIPPED in v3 (council: skip until SFT+DPO baseline works)
    # "grpo_max_steps":    200,
    # "grpo_lr":           5e-6,
    # "grpo_batch_size":   1,
    # "grpo_grad_accum":   4,
    # "grpo_num_generations": 4,
    # "grpo_max_completion_length": 2048,
    # "grpo_output_dir":   "./grpo_checkpoint",

    # ── Output / Push ─────────────────────────────────────────────────────────
    "hf_model_repo":     "OliverSlivka/qwen2.5-7b-itemset-extractor",
    "push_to_hub":       True,
}

print("✅ CONFIG loaded (v3.11 — versioned HF repos)")
print(f"   Model: {CONFIG['base_model']}")
print(f"   Seq length: {CONFIG['max_seq_length']}")
print(f"   LoRA: r={CONFIG['lora_r']}, alpha={CONFIG['lora_alpha']}, dropout={CONFIG['lora_dropout']}")
print(f"   SFT: {CONFIG['sft_epochs']} epochs @ lr={CONFIG['sft_lr']}")
print(f"   DPO: {CONFIG['dpo_epochs']} epoch @ lr={CONFIG['dpo_lr']}, beta={CONFIG['dpo_beta']}")
print(f"   GRPO: SKIPPED (v3)")
print(f"   Dataset: {CONFIG['hf_dataset']}")

In [ ]:
# ── Cell 3: GPU check + imports ───────────────────────────────────────────────
# Disable TF/JAX BEFORE any transformers/datasets import.
# The TLJH system has TensorFlow + Keras 3 installed; transformers detects TF
# as available and tries to import TFPreTrainedModel → Keras 3 crash.
#
# TLJH fix: choose a valid visible GPU BEFORE importing torch.
# This avoids stale CUDA_VISIBLE_DEVICES mappings like "0,1,2,3" when only
# 3 GPUs are actually visible to the notebook process.
import os, sys, json, re, subprocess, warnings

os.environ["USE_TF"] = "0"
os.environ["USE_JAX"] = "0"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

# Suppress noisy third-party deprecations we cannot fix from notebook code.
warnings.filterwarnings(
    "ignore",
    message=r"The attention mask API under `transformers\.modeling_attn_mask_utils`.*",
    category=FutureWarning,
)

# Torch must not be imported before GPU visibility is fixed.
if "torch" in sys.modules:
    raise RuntimeError(
        "Torch is already imported in this kernel before GPU selection. "
        "Restart the kernel, then run from Cell 2 again."
    )


def _query_gpus():
    cmd = [
        "nvidia-smi",
        "--query-gpu=index,name,memory.total,memory.free",
        "--format=csv,noheader,nounits",
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"nvidia-smi failed: {result.stderr.strip()[:200]}")

    gpus = []
    for line in result.stdout.strip().splitlines():
        if not line.strip():
            continue
        parts = [part.strip() for part in line.split(",")]
        if len(parts) != 4:
            continue
        gpu_index, name, total_mem, free_mem = parts
        gpus.append({
            "index": int(gpu_index),
            "name": name,
            "total_mem": int(total_mem),
            "free_mem": int(free_mem),
        })
    if not gpus:
        raise RuntimeError("No GPUs found via nvidia-smi")
    return gpus


def _parse_visible_devices(raw_value):
    if not raw_value:
        return None
    raw_value = raw_value.strip()
    if raw_value in {"", "NoDevFiles"}:
        return None
    parsed = []
    for token in raw_value.split(","):
        token = token.strip()
        if not token:
            continue
        if not token.isdigit():
            return None
        parsed.append(int(token))
    return parsed or None


gpus = _query_gpus()
current_visible = _parse_visible_devices(os.environ.get("CUDA_VISIBLE_DEVICES", ""))
available_indexes = {gpu["index"] for gpu in gpus}

# If inherited CUDA_VISIBLE_DEVICES is invalid for this process, replace it.
if current_visible and all(idx in available_indexes for idx in current_visible):
    selected_gpu = current_visible[0]
    visibility_reason = f"using existing CUDA_VISIBLE_DEVICES={os.environ['CUDA_VISIBLE_DEVICES']}"
else:
    best_gpu = max(gpus, key=lambda gpu: gpu["free_mem"])
    selected_gpu = best_gpu["index"]
    os.environ["CUDA_VISIBLE_DEVICES"] = str(selected_gpu)
    visibility_reason = (
        f"set CUDA_VISIBLE_DEVICES={selected_gpu} "
        f"(auto-selected GPU with most free VRAM)"
    )

print(f"🖥️  GPU visibility: {visibility_reason}")
for gpu in sorted(gpus, key=lambda item: item['index']):
    marker = "← selected" if gpu["index"] == selected_gpu else ""
    print(
        f"   GPU {gpu['index']}: {gpu['name']} | "
        f"free {gpu['free_mem']} MiB / total {gpu['total_mem']} MiB {marker}"
    )

import gc, torch
from datasets import load_dataset
from huggingface_hub import login

# HF login
hf_token = CONFIG["hf_token"] or os.environ.get("HF_TOKEN", "")
if hf_token:
    login(token=hf_token)
    print("✅ HuggingFace logged in")
else:
    print("⚠️  No HF token — will prompt for login if needed")
    try:
        login()
    except Exception:
        print("   Skipping login — set hf_token in CONFIG or run `huggingface-cli login`")

# GPU info (torch now only sees the selected visible GPU as cuda:0)
if torch.cuda.is_available():
    torch.cuda.init()
    print(f"\n✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    !nvidia-smi --query-gpu=index,name,memory.total,memory.free --format=csv,noheader
else:
    raise RuntimeError("❌ No GPU found — connect a GPU runtime before continuing.")

In [ ]:
# ── Cell 4: Load datasets from HuggingFace ────────────────────────────────────
# The v2 dataset has 3 configs: sft, dpo, grpo
print("📂 Loading datasets from HuggingFace Hub...")

sft_dataset  = load_dataset(CONFIG["hf_dataset"], "sft")
dpo_dataset  = load_dataset(CONFIG["hf_dataset"], "dpo")
grpo_dataset = load_dataset(CONFIG["hf_dataset"], "grpo")

print(f"✅ SFT:  {len(sft_dataset['train']):>4d} train / {len(sft_dataset['validation']):>3d} val")
print(f"✅ DPO:  {len(dpo_dataset['train']):>4d} train / {len(dpo_dataset['validation']):>3d} val")
print(f"✅ GRPO: {len(grpo_dataset['train']):>4d} train / {len(grpo_dataset['validation']):>3d} val")

In [ ]:
# ── Cell 5: Quick data preview ────────────────────────────────────────────────
example = sft_dataset["train"][0]
print("═" * 60)
print("SFT Example (messages format):")
print("═" * 60)
for msg in example["messages"]:
    role = msg["role"].upper()
    content = msg["content"]
    if len(content) > 300:
        content = content[:300] + f"... ({len(content)} chars total)"
    print(f"\n[{role}]")
    print(content)

print("\n" + "═" * 60)
print("DPO Example (prompt/chosen/rejected):")
print("═" * 60)
dpo_ex = dpo_dataset["train"][0]
print(f"Prompt: {len(dpo_ex['prompt'])} messages")
print(f"Chosen: {len(dpo_ex['chosen'][0]['content'])} chars")
print(f"Rejected: {len(dpo_ex['rejected'][0]['content'])} chars")

---
## Phase 1 — SFT with Chain-of-Thought 🎓

Teaches the model to **reason step-by-step** using `<think>` tags with **v3 concise format**:
```
<think>
Dataset: 7 rows × 12 cols, min_support=3
## SCAN 1: Singles by column
age: 15=3(R1,R2,R7)✓ | 16=2✗ | 17=4(R3,R4,R5,R6)✓
medu: 4=5(R1,R3,R5,R6,R7)✓ | 1=2✗
## SCAN 2: Pairs
{age:15,medu:4}: R1,R7 → 2✗
{age:17,medu:4}: R3,R5,R6 → 3✓
## RESULT SUMMARY: 5 singles + 3 pairs + 1 triple = 9 itemsets
</think>
[{"itemset": [...], "count": N, "rows": ["Row 1", ...]}]
```

**v3 changes from council:**
- Concise CoT format (column-grouped, no evidence_rows in think)
- 3 epochs at lr=1e-4 (was 2 epochs at 2e-4)
- LoRA r=32, alpha=64 (ratio 2.0), dropout=0.05
- `load_best_model_at_end=True` with eval every 50 steps
- Uses `train_on_responses_only` — only trains on assistant content

In [ ]:
# ── Cell 7: Load model + LoRA ─────────────────────────────────────────────────
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = CONFIG["base_model"],
    max_seq_length = CONFIG["max_seq_length"],
    load_in_4bit   = CONFIG["load_in_4bit"],
    dtype          = None,  # auto: bfloat16 on Ampere+, float16 on older
)

model = FastLanguageModel.get_peft_model(
    model,
    r                         = CONFIG["lora_r"],
    lora_alpha                = CONFIG["lora_alpha"],
    target_modules            = CONFIG["lora_target_modules"],
    lora_dropout              = CONFIG["lora_dropout"],  # v3: 0.05 (was 0)
    bias                      = "none",
    use_gradient_checkpointing = "unsloth",
    random_state              = 42,
)

model.print_trainable_parameters()

In [ ]:
# ── Cell 8: SFT training ─────────────────────────────────────────────────────
# v3.7 FIX: audit actual token lengths before training so we do not silently
# teach the model truncated <think> / JSON completions.
from math import ceil
import os
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

# Pre-format: apply chat template to messages → plain text column.
# train_on_responses_only needs a text dataset (not raw messages dict),
# otherwise _tokenize_fn inside unsloth_zoo gets an empty list → IndexError.
def apply_template(examples):
    return {
        "text": [
            tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
            for msgs in examples["messages"]
        ]
    }


def add_token_lengths(examples):
    tokenized = tokenizer(
        examples["text"],
        add_special_tokens=False,
        truncation=False,
    )
    return {"token_length": [len(ids) for ids in tokenized["input_ids"]]}

map_num_proc = max(1, min(8, os.cpu_count() or 1, len(sft_dataset["train"]), len(sft_dataset["validation"])))

sft_train_fmt = sft_dataset["train"].map(
    apply_template,
    batched=True,
    num_proc=map_num_proc,
    remove_columns=sft_dataset["train"].column_names,
    desc="Formatting SFT train chat templates",
)
sft_val_fmt = sft_dataset["validation"].map(
    apply_template,
    batched=True,
    num_proc=map_num_proc,
    remove_columns=sft_dataset["validation"].column_names,
    desc="Formatting SFT validation chat templates",
)

sft_train_fmt = sft_train_fmt.map(add_token_lengths, batched=True, num_proc=map_num_proc, desc="Counting SFT train tokens")
sft_val_fmt = sft_val_fmt.map(add_token_lengths, batched=True, num_proc=map_num_proc, desc="Counting SFT validation tokens")

train_over_limit = sum(length > CONFIG["max_seq_length"] for length in sft_train_fmt["token_length"])
val_over_limit = sum(length > CONFIG["max_seq_length"] for length in sft_val_fmt["token_length"])
train_max_len = max(sft_train_fmt["token_length"])
val_max_len = max(sft_val_fmt["token_length"])

print(f"✅ Dataset formatted — train: {len(sft_train_fmt)}, val: {len(sft_val_fmt)}")
print(f"   Sample (first 200 chars): {sft_train_fmt[0]['text'][:200]}")
print(f"   Train max token length: {train_max_len}")
print(f"   Val max token length:   {val_max_len}")
print(f"   Over limit (> {CONFIG['max_seq_length']}) — train: {train_over_limit}, val: {val_over_limit}")

assert train_over_limit == 0 and val_over_limit == 0, (
    f"❌ Some SFT examples exceed max_seq_length={CONFIG['max_seq_length']} and would be truncated during training. "
    f"Increase CONFIG['max_seq_length'] or regenerate/filter the dataset first. "
    f"Counts — train: {train_over_limit}, val: {val_over_limit}"
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
if tokenizer.bos_token_id is None and getattr(model.config, "bos_token_id", None) is not None:
    tokenizer.bos_token_id = model.config.bos_token_id

sft_total_steps = ceil(
    len(sft_train_fmt) / (CONFIG["sft_batch_size"] * CONFIG["sft_grad_accum"])
 ) * CONFIG["sft_epochs"]
sft_warmup_steps = max(1, ceil(sft_total_steps * CONFIG["sft_warmup_ratio"]))
print(f"   SFT optimizer steps: {sft_total_steps}, warmup_steps: {sft_warmup_steps}")

sft_trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = sft_train_fmt.remove_columns(["token_length"]),
    eval_dataset    = sft_val_fmt.remove_columns(["token_length"]),
    args            = SFTConfig(
        dataset_text_field          = "text",
        dataset_num_proc            = map_num_proc,
        max_seq_length              = CONFIG["max_seq_length"],
        num_train_epochs            = CONFIG["sft_epochs"],
        per_device_train_batch_size = CONFIG["sft_batch_size"],
        gradient_accumulation_steps = CONFIG["sft_grad_accum"],
        learning_rate               = CONFIG["sft_lr"],
        lr_scheduler_type           = "cosine",
        warmup_steps                = sft_warmup_steps,
        weight_decay                = CONFIG["sft_weight_decay"],
        optim                       = "paged_adamw_8bit",
        bf16                        = True,
        fp16                        = False,
        logging_steps               = 10,
        eval_strategy               = "steps",
        eval_steps                  = 50,
        save_strategy               = "steps",
        save_steps                  = 50,
        save_total_limit            = 3,
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_loss",
        output_dir                  = CONFIG["sft_output_dir"],
        report_to                   = "none",
        seed                        = 42,
    ),
)

# Only train on assistant responses — mask system + user tokens
sft_trainer = train_on_responses_only(
    sft_trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

print("🎓 Starting SFT training with Chain-of-Thought (v3.7 truncation-safe config)...")
print(f"   Epochs: {CONFIG['sft_epochs']}, LR: {CONFIG['sft_lr']}, Warmup steps: {sft_warmup_steps}")
print(f"   Optimizer: paged_adamw_8bit")
sft_result = sft_trainer.train()
print(f"✅ SFT done! Final loss: {sft_result.training_loss:.4f}")

In [ ]:
# ── Cell 8b: Label masking verification (v3.2 — diamond knowledge) ───────────
# WHY: train_on_responses_only() silently fails if instruction_part or
# response_part strings don't match the actual tokenized template. This cell
# decodes one training batch to VISUALLY CONFIRM that only assistant responses
# are trained on (-100 = masked, real IDs = trained).
# SOURCE: Unsloth Thinking + Coder notebooks — all use this verification pattern.

# Use the original formatted dataset, not sft_trainer.train_dataset, because the
# trainer may replace/remove the raw "text" column during internal preprocessing.
sample_text = sft_train_fmt[0]["text"]
processor = sft_trainer.processing_class

tokenized = processor(
    sample_text,
    truncation=True,
    max_length=CONFIG["max_seq_length"],
    return_tensors="pt",
)

# Pull one actual training batch after train_on_responses_only masking is active.
sample_batch = sft_trainer.get_train_dataloader()
for batch_check in sample_batch:
    labels = batch_check["labels"][0].detach().cpu()
    break

# Decode: replace -100 (masked) with a visible marker.
space_tokens = processor.encode(" ", add_special_tokens=False)
space_id = space_tokens[0] if space_tokens else processor.eos_token_id
masked_labels = labels.clone()
masked_labels[masked_labels == -100] = space_id
decoded_trained = processor.decode(masked_labels, skip_special_tokens=False)

# Summary stats
total_tokens = len(labels)
masked_count = (labels == -100).sum().item()
trained_count = total_tokens - masked_count

print("=" * 60)
print("🔍 LABEL MASKING VERIFICATION (v3.2 — diamond knowledge)")
print("=" * 60)
print(f"Raw sample chars: {len(sample_text)}")
print(f"Tokenized sample length: {tokenized['input_ids'].shape[-1]}")
print(f"Total tokens in training batch: {total_tokens}")
print(f"Masked (prompt/system): {masked_count} ({masked_count/total_tokens*100:.1f}%)")
print(f"Trained (assistant response): {trained_count} ({trained_count/total_tokens*100:.1f}%)")
print()
print("Visible decoded labels (masked tokens rendered as spaces):")
print("-" * 60)
print(decoded_trained[:500])
print("-" * 60)

# Sanity checks
assert trained_count > 0, "❌ CRITICAL: No tokens are being trained on! Masking is broken."
assert masked_count > 0, "❌ CRITICAL: No tokens are masked! train_on_responses_only() had no effect."
assert trained_count / total_tokens < 0.85, "⚠️ WARNING: >85% of tokens trained — masking may not be working correctly."
print(f"\n✅ Label masking looks correct — {trained_count} response tokens trained, {masked_count} prompt tokens masked.")

In [ ]:
# ── Cell 8c: Inline inference utilities (v3.14) ─────────────────────────────
# These are inlined from src/evaluation/inference_utils.py so the notebook is
# fully self-contained on TLJH (where the full repo is NOT cloned).
# Used by Cell 9 (SFT gate) and Cell 19 (quick inference test).
#
# v3.14 FIXES:
#   1. generate_two_phase() now accepts and forwards attention_mask so
#      generation does not emit pad/eos ambiguity warnings.
#   2. Cell 9 can request deterministic decoding for the gate via
#      think_do_sample=False and json_do_sample=False.
#   3. Phase 2 builds an explicit attention mask for the re-encoded JSON prompt.

import re, json, torch
from transformers import StoppingCriteria, StoppingCriteriaList


class ThinkStoppingCriteria(StoppingCriteria):
    """Stop generation when </think> token sequence is produced."""

    def __init__(self, tokenizer, max_think_tokens: int = 3000):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_think_tokens = max_think_tokens
        self.stop_ids = tokenizer.encode("</think>", add_special_tokens=False)
        self.stop_len = len(self.stop_ids)
        self._generated_count = 0

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        self._generated_count += 1
        if self._generated_count >= self.max_think_tokens:
            return True
        if input_ids.shape[1] >= self.stop_len:
            last_tokens = input_ids[0, -self.stop_len:].tolist()
            if last_tokens == self.stop_ids:
                return True
        return False

    def reset(self):
        self._generated_count = 0


class RepetitionDetector(StoppingCriteria):
    """Stop generation if the model enters a repetition loop."""

    def __init__(self, tokenizer, max_repeats: int = 3, check_every: int = 50):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_repeats = max_repeats
        self.check_every = check_every
        self._step = 0

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        self._step += 1
        if self._step % self.check_every != 0:
            return False
        text = self.tokenizer.decode(input_ids[0, -500:], skip_special_tokens=True)
        lines = [line.strip() for line in text.split("\n") if line.strip()]
        if len(lines) < self.max_repeats * 2:
            return False
        recent = lines[-self.max_repeats:]
        if len(set(recent)) == 1 and len(recent) >= self.max_repeats:
            return True
        return False

    def reset(self):
        self._step = 0


def generate_two_phase(
    model,
    tokenizer,
    input_ids: torch.LongTensor,
    attention_mask: torch.LongTensor | None = None,
    think_temperature: float = 0.3,
    json_temperature: float = 0.05,
    think_max_tokens: int = 2000,
    json_max_tokens: int = 1500,
    top_k: int = 50,
    top_p: float = 0.90,
    think_do_sample: bool = True,
    json_do_sample: bool = True,
) -> str:
    """
    Two-phase generation: <think> phase + JSON phase.
    Phase 1: reasoning at higher temp → stop at </think>
    Phase 2: JSON at very low temp for precision.
    """
    device = input_ids.device

    think_stopper = ThinkStoppingCriteria(tokenizer, max_think_tokens=think_max_tokens)
    rep_detector = RepetitionDetector(tokenizer, max_repeats=3)

    phase1_kwargs = {
        "input_ids": input_ids,
        "max_new_tokens": think_max_tokens,
        "do_sample": think_do_sample,
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
        "stopping_criteria": StoppingCriteriaList([think_stopper, rep_detector]),
    }
    if attention_mask is not None:
        phase1_kwargs["attention_mask"] = attention_mask
    if think_do_sample:
        phase1_kwargs.update({
            "temperature": think_temperature,
            "top_k": top_k,
            "top_p": top_p,
        })

    with torch.no_grad():
        think_output = model.generate(**phase1_kwargs)

    think_text = tokenizer.decode(think_output[0][input_ids.shape[1]:], skip_special_tokens=True)

    if "</think>" in think_text:
        think_text = think_text.split("</think>")[0] + "</think>\n"
    else:
        lines = think_text.split("\n")
        last_good = len(lines) - 1
        for i in range(len(lines) - 1, -1, -1):
            if any(marker in lines[i] for marker in ["✓", "✗", "##", "RESULT", "FREQUENT", "SCAN"]):
                last_good = i
                break
        think_text = "\n".join(lines[:last_good + 1]) + "\n</think>\n"

    prompt_text = tokenizer.decode(input_ids[0], skip_special_tokens=False)
    json_prompt = prompt_text + think_text + "["
    json_inputs = tokenizer(json_prompt, return_tensors="pt").to(device)

    phase2_kwargs = {
        "input_ids": json_inputs["input_ids"],
        "attention_mask": json_inputs["attention_mask"],
        "max_new_tokens": json_max_tokens,
        "do_sample": json_do_sample,
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    if json_do_sample:
        phase2_kwargs.update({
            "temperature": json_temperature,
            "top_k": 20,
            "top_p": 0.95,
        })

    with torch.no_grad():
        json_output = model.generate(**phase2_kwargs)

    json_text = tokenizer.decode(
        json_output[0][json_inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    full_response = think_text
    if not full_response.rstrip().endswith("</think>"):
        full_response = full_response.rstrip() + "\n</think>\n"
    full_response += "[" + json_text

    return full_response


def extract_and_repair_json(raw_output: str) -> tuple:
    """Extract JSON array from model output, handling various failure modes."""
    has_think = "<think>" in raw_output and "</think>" in raw_output
    json_text = raw_output

    if has_think:
        last_think_idx = raw_output.rfind("</think>")
        json_text = raw_output[last_think_idx + len("</think>"):].strip()

    try:
        parsed = json.loads(json_text)
        if isinstance(parsed, list):
            return parsed, True, json_text
    except json.JSONDecodeError:
        pass

    match = re.search(r"\[.*?\]", json_text, re.DOTALL)
    if match:
        try:
            parsed = json.loads(match.group())
            if isinstance(parsed, list):
                return parsed, True, match.group()
        except json.JSONDecodeError:
            pass

    match = re.search(r"\[.*\]", json_text, re.DOTALL)
    if match:
        try:
            parsed = json.loads(match.group())
            if isinstance(parsed, list):
                return parsed, True, match.group()
        except json.JSONDecodeError:
            pass

    if has_think:
        last_think_idx = raw_output.rfind("</think>")
        after_last = raw_output[last_think_idx + len("</think>"):]
        match = re.search(r"\[.*\]", after_last, re.DOTALL)
        if match:
            try:
                parsed = json.loads(match.group())
                if isinstance(parsed, list):
                    return parsed, True, match.group()
            except json.JSONDecodeError:
                pass

    return [], False, json_text


print("✅ Inference utilities defined inline (v3.14):")
print("   • ThinkStoppingCriteria — stops at </think>")
print("   • RepetitionDetector — catches loops")
print("   • generate_two_phase() — attention-mask aware, gate can run deterministically")
print("   • extract_and_repair_json() — robust JSON parser")

In [ ]:
# ── Cell 9: Save SFT + format verification gate (v3.15) ─────────────────────
# v3.14 FIX: Run the SFT gate with HF-native SDPA inference instead of
# Unsloth's patched generate(). On TLJH H200, Unsloth may fall back to
# xformers for inference and crash inside the gate with rotary-shape mismatch
# errors even though SFT training itself completed successfully.
#
# v3.15 budget rebalance:
#   • slightly higher requested completion budget (2.3x target tokens)
#   • smaller think share (~48%)
#   • larger JSON floor (768 tokens when budget allows)
#   • low-budget fallback so think/json floors do not overcommit the context window
#
# Additional cleanup retained from v3.14:
#   • deterministic gate decoding (diagnostic-first)
#   • explicit attention_mask during generation
#   • clear max_length on generation_config to avoid max_new_tokens conflicts

from pathlib import Path
import gc, importlib, json, torch
from peft import PeftModel

sft_dir = Path(CONFIG["sft_output_dir"])
raw_out_dir = Path("eval_raw_capture") / "sft_gate"
raw_out_dir.mkdir(parents=True, exist_ok=True)

if "model" in dir() and model is not None:
    model.save_pretrained(CONFIG["sft_output_dir"])
    if "tokenizer" in dir() and tokenizer is not None:
        tokenizer.save_pretrained(CONFIG["sft_output_dir"])
    print(f"💾 SFT adapter saved → {CONFIG['sft_output_dir']}")
elif not sft_dir.exists():
    raise RuntimeError(
        "SFT model is not in memory and no saved checkpoint exists yet. "
        "Run Cell 7 → Cell 8 first, then rerun this cell."
    )
else:
    print(f"♻️ Using existing saved SFT adapter → {CONFIG['sft_output_dir']}")

if "model" in dir() and model is not None:
    del model
if "sft_trainer" in dir():
    del sft_trainer
gc.collect()
torch.cuda.empty_cache()

print("\n" + "=" * 60)
print("🚦 SFT FORMAT VERIFICATION GATE (v3.15 — HF native SDPA)")
print("=" * 60)

import transformers.models.qwen2.modeling_qwen2 as _qwen2_mod
_qwen2_mod = importlib.reload(_qwen2_mod)
_Qwen2ForCausalLM = _qwen2_mod.Qwen2ForCausalLM

import peft.peft_model as _peft_mod
_peft_mod = importlib.reload(_peft_mod)
_PeftModel = _peft_mod.PeftModel

from transformers import AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

gate_model = _Qwen2ForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct",
    quantization_config=bnb_config,
    attn_implementation="sdpa",
    dtype=torch.bfloat16,
    device_map={"": 0},
)

gate_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
if gate_tokenizer.pad_token_id is None:
    gate_tokenizer.pad_token = gate_tokenizer.eos_token
if gate_tokenizer.bos_token_id is None and getattr(gate_model.config, "bos_token_id", None) is not None:
    gate_tokenizer.bos_token_id = gate_model.config.bos_token_id

gate_model = _PeftModel.from_pretrained(gate_model, CONFIG["sft_output_dir"])
gate_model.eval()
if hasattr(gate_model, "generation_config") and hasattr(gate_model.generation_config, "max_length"):
    gate_model.generation_config.max_length = None
print("✅ Gate model loaded with HF-native SDPA attention")

if "sft_val_fmt" not in dir() or sft_val_fmt is None:
    if "sft_dataset" not in dir():
        raise RuntimeError(
            "sft_val_fmt is missing and sft_dataset is not loaded. "
            "Run Cell 4 first, then rerun this cell."
        )

    def _gate_apply_template(examples):
        return {
            "text": [
                gate_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
                for msgs in examples["messages"]
            ]
        }

    sft_val_fmt = sft_dataset["validation"].map(
        _gate_apply_template,
        batched=True,
        remove_columns=sft_dataset["validation"].column_names,
    )
    print("♻️ Rebuilt sft_val_fmt from sft_dataset['validation'] for gate rerun")

PROMPT_MARKER = "<|im_start|>assistant\n"


def estimate_gate_targets(sample_text, user_end):
    target_response = sample_text[user_end + len(PROMPT_MARKER):]
    target_token_count = len(
        gate_tokenizer(target_response, add_special_tokens=False)["input_ids"]
    )
    return target_response, target_token_count


def build_two_phase_budgets(prompt_token_count, target_token_count):
    available_completion_budget = max(1024, CONFIG["max_seq_length"] - prompt_token_count - 64)
    requested_budget = max(1792, int(target_token_count * 2.3))
    total_budget = min(available_completion_budget, requested_budget)

    if total_budget >= 1536:
        json_budget = max(768, int(total_budget * 0.52))
        json_budget = min(json_budget, total_budget - 512)
        think_budget = total_budget - json_budget
        think_budget = min(1800, think_budget)
        json_budget = total_budget - think_budget
    else:
        json_budget = max(512, total_budget // 2)
        think_budget = total_budget - json_budget

    return available_completion_budget, requested_budget, total_budget, think_budget, json_budget


gate_pass = 0
gate_total = min(2, len(sft_val_fmt))
hit_budget_count = 0
for i in range(gate_total):
    sample_text = sft_val_fmt[i]["text"]
    user_end = sample_text.find(PROMPT_MARKER)
    if user_end == -1:
        continue

    prompt_text = sample_text[:user_end] + PROMPT_MARKER
    target_response, target_token_count = estimate_gate_targets(sample_text, user_end)

    inputs = gate_tokenizer(prompt_text, return_tensors="pt").to(gate_model.device)
    prompt_token_count = inputs["input_ids"].shape[1]
    (
        available_completion_budget,
        requested_budget,
        total_budget,
        think_budget,
        json_budget,
    ) = build_two_phase_budgets(prompt_token_count, target_token_count)

    generated = generate_two_phase(
        gate_model,
        gate_tokenizer,
        inputs["input_ids"],
        attention_mask   = inputs.get("attention_mask"),
        think_temperature = 0.3,
        json_temperature  = 0.05,
        think_max_tokens  = think_budget,
        json_max_tokens   = json_budget,
        top_k             = 50,
        top_p             = 0.90,
        think_do_sample   = False,
        json_do_sample    = False,
    )

    generated_token_count = len(
        gate_tokenizer(generated, add_special_tokens=False)["input_ids"]
    )
    hit_budget = generated_token_count >= total_budget
    if hit_budget:
        hit_budget_count += 1

    raw_path = raw_out_dir / f"sample_{i+1}.txt"
    meta_path = raw_out_dir / f"sample_{i+1}_meta.json"
    raw_path.write_text(generated, encoding="utf-8")

    think_open = "<think>" in generated
    think_close = "</think>" in generated
    parsed, parse_ok, parsed_json_text = extract_and_repair_json(generated)
    has_json = bool(parse_ok and isinstance(parsed, list))

    has_colval = False
    parsed_itemsets = []
    if has_json:
        parsed_itemsets = [
            item for item in parsed
            if isinstance(item, dict) and isinstance(item.get("itemset"), list)
        ]
        if parsed_itemsets:
            has_colval = any(
                ":" in str(token)
                for item in parsed_itemsets[:5]
                for token in item.get("itemset", [])
            )

    meta = {
        "sample_index": i + 1,
        "prompt_token_count": int(prompt_token_count),
        "target_token_count": int(target_token_count),
        "available_completion_budget": int(available_completion_budget),
        "requested_budget": int(requested_budget),
        "total_budget": int(total_budget),
        "think_budget": int(think_budget),
        "json_budget": int(json_budget),
        "generated_token_count": int(generated_token_count),
        "hit_budget": bool(hit_budget),
        "think_open": bool(think_open),
        "think_close": bool(think_close),
        "has_json_after_think": bool(has_json),
        "has_colval_in_json": bool(has_colval),
        "raw_path": str(raw_path),
    }
    meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")

    status = "✅" if (think_open and think_close and has_json and has_colval) else "⚠️"
    if think_open and think_close and has_json and has_colval:
        gate_pass += 1

    print(f"\n{status} Sample {i+1}/{gate_total}:")
    print(f"   Prompt length: {prompt_token_count} tokens")
    print(f"   Target completion length: {target_token_count} tokens")
    print(f"   Available completion budget: {available_completion_budget} tokens")
    print(f"   Requested budget: {requested_budget} tokens")
    print(f"   Total generation budget: {total_budget} tokens")
    print(f"   Think budget: {think_budget} tokens")
    print(f"   JSON budget: {json_budget} tokens")
    print(f"   Generated length: {generated_token_count} tokens / {len(generated)} chars")
    print(f"   Hit token budget: {'⚠️ YES' if hit_budget else '✅ no'}")
    print(f"   <think> opened: {'✅' if think_open else '❌'}")
    print(f"   </think> closed: {'✅' if think_close else '❌'}")
    print(f"   Valid JSON array after </think>: {'✅' if has_json else '❌'}")
    print(f"   col:value format in parsed JSON: {'✅' if has_colval else '❌'}")
    print(f"   Raw output saved to: {raw_path}")
    print(f"   Metadata saved to: {meta_path}")
    print("   Output head:")
    print("   " + generated[:220].replace("\n", "\n   "))
    print("   Output tail:")
    print("   " + generated[-220:].replace("\n", "\n   "))

    if parsed_json_text:
        print("   Parsed JSON head:")
        print("   " + parsed_json_text[:220].replace("\n", "\n   "))
        if parsed_itemsets:
            print(f"   First parsed itemset: {parsed_itemsets[0]}")
    else:
        if think_open and not think_close and hit_budget:
            print("   Parsed JSON head: <none — generation hit token budget before </think>>")
        elif think_open and not think_close:
            print("   Parsed JSON head: <none — model stopped early before </think>>")
        else:
            print("   Parsed JSON head: <none found>")

gate_ratio = gate_pass / gate_total if gate_total > 0 else 0
print(f"\n{'=' * 60}")
print(f"Gate result: {gate_pass}/{gate_total} passed ({gate_ratio*100:.0f}%)")
if gate_ratio >= 0.5:
    print("✅ GATE PASSED — SFT format looks correct under guarded two-phase decoding. Proceeding to DPO.")
else:
    print("⚠️  GATE WARNING: SFT format compliance remains low even under guarded two-phase decoding.")
    if hit_budget_count > 0:
        print("   At least one sample still hit the completion cap, so truncation remains a likely factor.")
    else:
        print("   Samples still fail without hitting the cap, which points to real model/data quality issues.")
    print("   Inspect the saved raw output and metadata files before concluding the adapter is bad.")
print("=" * 60)

del gate_model
gc.collect()
torch.cuda.empty_cache()
print(f"\n🧹 Memory freed. VRAM available: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.1f} GB")

---
## Phase 2 — DPO with Real LLM Failures 🎯

Uses **actual mistakes** from GPT-4.1-mini, GPT-4.1-nano, o4-mini, GPT-4o:
- **Chosen** = Apriori ground truth with CoT reasoning
- **Rejected** = Real LLM outputs that failed validation (99.5% error = hallucinated evidence rows)

Why real failures beat synthetic:
- Match the **actual error distribution** LLMs produce
- Model learns to avoid the exact mistakes it would naturally make
- 606+ real DPO pairs from 313 unique datasets

**v3 changes from council:**
- 1 epoch only (was 2) — DPO overfits quickly with real failures
- `warmup_ratio=0.10` (was 0.05)
- DPO continues training the existing SFT adapter weights
- Final artifact stays adapter-only: base model + one DPO adapter for inference

In [ ]:
# ── Cell 11: Reload base + load SFT adapter for DPO (v3.18) ─────────────────
# v3.12 CRITICAL FIX: Undo Unsloth's monkey-patches for clean DPO training.
#
# v3.18 FIX: Some TRL builds import optional helper stacks while loading
# DPOTrainer (for example mergekit, llm_blender, and weave). On TLJH, those
# extras are not installed and are not needed for DPO, so we force-disable
# their discovery before reloading the trainer modules.
#
# PROBLEM: Importing Unsloth (Cell 7 for SFT) permanently patches CLASS-level
# forward methods on Qwen2Model, Qwen2ForCausalLM, PeftModel, and hijacks
# DPOTrainer → UnslothDPOTrainer. These patches:
#   1. Route attention through xformers (fails on DPO's padded concat batches
#      with BMGHK 5D tensor shape from Qwen2.5's grouped-query attention)
#   2. Expect Unsloth-specific attributes (e.g., model.max_seq_length)
#   3. Persist even when loading new models via HF native AutoModelForCausalLM
#
# FIX: importlib.reload() the patched modules → fresh classes with original
# forward methods. Load model + adapter + trainer from reloaded modules.
# This gives us clean HF native behavior without a kernel restart.

import importlib, gc, sys, torch

# ── Step 1: Free SFT model ───────────────────────────────────────────────────
if 'model' in dir() and model is not None:
    del model
if 'sft_trainer' in dir():
    del sft_trainer
gc.collect()
torch.cuda.empty_cache()

# ── Step 2: Remove Unsloth import hooks from sys.meta_path ───────────────────
# Unsloth registers custom import finders that redirect trl.DPOTrainer →
# UnslothDPOTrainer. Removing them ensures reloads get standard classes.
_before = len(sys.meta_path)
sys.meta_path[:] = [
    finder for finder in sys.meta_path
    if not any('unsloth' in s.lower() for s in [
        str(getattr(type(finder), '__module__', '')),
        type(finder).__name__,
        str(getattr(finder, 'path', '')),
    ])
]
print(f"🔧 Removed {_before - len(sys.meta_path)} Unsloth import hook(s)")

# ── Step 3: Reload monkey-patched modules to restore original classes ─────────
print("📦 Reloading patched modules to restore originals...")

import transformers.models.qwen2.modeling_qwen2 as _qwen2_mod
_qwen2_mod = importlib.reload(_qwen2_mod)
_Qwen2ForCausalLM = _qwen2_mod.Qwen2ForCausalLM

import peft.peft_model as _peft_mod
_peft_mod = importlib.reload(_peft_mod)
_PeftModel = _peft_mod.PeftModel

import trl.import_utils as _trl_import_utils
_trl_import_utils.is_mergekit_available = lambda: False
_trl_import_utils.is_llm_blender_available = lambda: False
_trl_import_utils.is_weave_available = lambda: False

for stale_mod in [
    "trl.mergekit_utils",
    "trl.trainer.judges",
    "trl.trainer.callbacks",
    "trl.trainer.dpo_trainer",
]:
    if stale_mod in sys.modules:
        del sys.modules[stale_mod]

try:
    import trl.trainer.dpo_trainer as _dpo_mod
except ModuleNotFoundError as exc:
    if exc.name not in {"mergekit", "llm_blender", "weave"}:
        raise
    print(f"⚠️ TRL tried to import optional dependency '{exc.name}'; retrying with optional integrations disabled...")
    _trl_import_utils.is_mergekit_available = lambda: False
    _trl_import_utils.is_llm_blender_available = lambda: False
    _trl_import_utils.is_weave_available = lambda: False
    for stale_mod in [
        "trl.mergekit_utils",
        "trl.trainer.judges",
        "trl.trainer.callbacks",
        "trl.trainer.dpo_trainer",
    ]:
        if stale_mod in sys.modules:
            del sys.modules[stale_mod]
    _dpo_mod = importlib.import_module("trl.trainer.dpo_trainer")

_dpo_mod = importlib.reload(_dpo_mod)
_DPOTrainer = _dpo_mod.DPOTrainer

# Verify: forward methods must NOT be from Unsloth
for cls_obj, cls_name in [(_Qwen2ForCausalLM, "Qwen2ForCausalLM"),
                           (_PeftModel, "PeftModel")]:
    fwd_name = getattr(cls_obj.forward, '__qualname__', '')
    assert 'fast_forward' not in fwd_name and 'unsloth' not in fwd_name.lower(), \
        f"❌ {cls_name}.forward still patched: {fwd_name}"

print("   ✅ Qwen2ForCausalLM, PeftModel, DPOTrainer — originals restored")

# ── Step 4: Load base model with fresh class + SDPA ──────────────────────────
from transformers import BitsAndBytesConfig, AutoTokenizer

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("\n📦 Loading Qwen2.5-7B-Instruct (fresh class + SDPA attention)...")
model = _Qwen2ForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct",
    quantization_config=bnb_config,
    attn_implementation="sdpa",
    dtype=torch.bfloat16,
    device_map={"": 0},
)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
if tokenizer.bos_token_id is None and getattr(model.config, "bos_token_id", None) is not None:
    tokenizer.bos_token_id = model.config.bos_token_id

# ── Step 5: Load SFT adapter with fresh PeftModel ────────────────────────────
model = _PeftModel.from_pretrained(
    model,
    CONFIG["sft_output_dir"],
    is_trainable=True,
)

model.print_trainable_parameters()
print("\n✅ DPO model ready (v3.18: Unsloth patches removed, SDPA attention, optional TRL deps bypassed)")
print(f"   Attention: SDPA (PyTorch native — no xformers BMGHK issue)")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# ── Cell 12: DPO training with real failures (v3.14) ─────────────────────────
# v3.13: Uses _DPOTrainer from reloaded module (Cell 11), NOT Unsloth's
# patched UnslothDPOTrainer. Standard TRL DPOTrainer + SDPA model.
#
# v3.14 FIX: Some TRL versions expect model.warnings_issued to exist so they can
# suppress a FLOPs/token-estimation warning during DPO init. HF-native Qwen2 +
# PEFT may not define that attribute, so we create a compatibility shim first.
from math import ceil
import os
from trl import DPOConfig

# Format DPO data: apply chat template to prompt, extract content from chosen/rejected
def format_dpo(examples):
    prompts, chosens, rejecteds = [], [], []
    for prompt, chosen, rejected in zip(examples["prompt"], examples["chosen"], examples["rejected"]):
        prompts.append(
            tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
        )
        chosens.append(chosen[0]["content"])
        rejecteds.append(rejected[0]["content"])
    return {"prompt": prompts, "chosen": chosens, "rejected": rejecteds}

dpo_map_num_proc = max(1, min(8, os.cpu_count() or 1, len(dpo_dataset["train"]), len(dpo_dataset["validation"])))

dpo_train = dpo_dataset["train"].map(
    format_dpo,
    batched=True,
    num_proc=dpo_map_num_proc,
    remove_columns=dpo_dataset["train"].column_names,
    desc="Formatting DPO train pairs",
)
dpo_val = dpo_dataset["validation"].map(
    format_dpo,
    batched=True,
    num_proc=dpo_map_num_proc,
    remove_columns=dpo_dataset["validation"].column_names,
    desc="Formatting DPO validation pairs",
)

dpo_total_steps = ceil(
    len(dpo_train) / (CONFIG["dpo_batch_size"] * CONFIG["dpo_grad_accum"])
 ) * CONFIG["dpo_epochs"]
dpo_warmup_steps = max(1, ceil(dpo_total_steps * 0.10))
print(f"   DPO optimizer steps: {dpo_total_steps}, warmup_steps: {dpo_warmup_steps}")

if not hasattr(model, "warnings_issued") or model.warnings_issued is None:
    model.warnings_issued = {}
base_dpo_model = getattr(model, "base_model", None)
if base_dpo_model is not None and (not hasattr(base_dpo_model, "warnings_issued") or base_dpo_model.warnings_issued is None):
    base_dpo_model.warnings_issued = {}
wrapped_dpo_model = getattr(base_dpo_model, "model", None) if base_dpo_model is not None else None
if wrapped_dpo_model is not None and (not hasattr(wrapped_dpo_model, "warnings_issued") or wrapped_dpo_model.warnings_issued is None):
    wrapped_dpo_model.warnings_issued = {}

dpo_trainer = _DPOTrainer(
    model            = model,
    processing_class = tokenizer,
    train_dataset    = dpo_train,
    eval_dataset     = dpo_val,
    args             = DPOConfig(
        beta                          = CONFIG["dpo_beta"],
        max_length                    = CONFIG["max_seq_length"],
        max_prompt_length             = CONFIG["max_seq_length"] // 2,
        num_train_epochs              = CONFIG["dpo_epochs"],
        per_device_train_batch_size   = CONFIG["dpo_batch_size"],
        gradient_accumulation_steps   = CONFIG["dpo_grad_accum"],
        learning_rate                 = CONFIG["dpo_lr"],
        lr_scheduler_type             = "cosine",
        warmup_steps                  = dpo_warmup_steps,
        optim                         = "paged_adamw_8bit",
        bf16                          = True,
        fp16                          = False,
        gradient_checkpointing        = True,
        gradient_checkpointing_kwargs = {"use_reentrant": False},
        logging_steps                 = 10,
        eval_strategy                 = "epoch",
        save_strategy                 = "epoch",
        save_total_limit              = 2,
        output_dir                    = CONFIG["dpo_output_dir"],
        report_to                     = "none",
        seed                          = 42,
    ),
)

print("🎯 Starting DPO training (v3.14: unpatched DPOTrainer + SDPA + warnings_issued shim)...")
dpo_result = dpo_trainer.train()
print(f"✅ DPO done! Final loss: {dpo_result.training_loss:.4f}")

In [ ]:
# ── Cell 13: Save DPO checkpoint (ADAPTER ONLY — v3 fix) ────────────────────
# v3: Adapter-only save. NEVER use merged_4bit_forced.
model.save_pretrained(CONFIG["dpo_output_dir"])
tokenizer.save_pretrained(CONFIG["dpo_output_dir"])
print(f"💾 DPO adapter saved → {CONFIG['dpo_output_dir']}")

# v3: GRPO is SKIPPED — jump directly to Cell 19 (inference test)
# If you want to try GRPO in future, uncomment the GRPO cells below.
print("\n📌 v3: GRPO skipped (council recommendation). Proceeding to inference test.")
print("   Next: Run Cell 19 (inference test) → Cell 20 (push to hub)")

---
## Phase 3 — GRPO with Apriori Reward 🔬 *(SKIPPED in v3)*

**⚠️ v3 Council Decision: SKIP GRPO** until SFT+DPO baseline achieves F1 ≥ 0.60.

Rationale (unanimous from 4 frontier models):
- GRPO adds complexity and instability without proven benefit for this task
- SFT+DPO should be sufficient to reach 80% F1 target
- GRPO can be added in v4 if needed as a refinement pass

**If you want to try GRPO anyway**, uncomment the cells below. But run eval on DPO first!

In [ ]:
# ── Cell 15: GRPO reward functions ───────────────────────────────────────────

def _extract_json(text):
    """Extract JSON array from model response, handling <think> tags."""
    # Strip <think> block if present
    if "</think>" in text:
        text = text.split("</think>", 1)[-1].strip()
    # Find JSON array
    m = re.search(r'\[.*\]', text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group())
        except json.JSONDecodeError:
            return None
    return None


def json_format_reward(completions, **kwargs):
    """Reward for valid JSON with correct schema: itemset, count, rows."""
    rewards = []
    for text in completions:
        parsed = _extract_json(text)
        if parsed is None:
            rewards.append(0.0)
        elif not isinstance(parsed, list) or len(parsed) == 0:
            rewards.append(0.2)
        elif all(
            isinstance(x, dict) and "itemset" in x and "count" in x and "rows" in x
            for x in parsed
        ):
            rewards.append(1.0)
        elif all(isinstance(x, dict) and "itemset" in x for x in parsed):
            rewards.append(0.5)
        else:
            rewards.append(0.2)
    return rewards


def itemset_f1_reward(completions, ground_truth, **kwargs):
    """F1 score of predicted itemsets vs Apriori ground truth."""
    rewards = []
    for text, gt_str in zip(completions, ground_truth):
        predicted = _extract_json(text)
        try:
            gt = json.loads(gt_str)
        except (json.JSONDecodeError, TypeError):
            rewards.append(0.0)
            continue

        if predicted is None:
            rewards.append(0.0)
            continue

        pred_sets = {frozenset(x["itemset"]) for x in predicted if isinstance(x, dict) and "itemset" in x}
        true_sets = {frozenset(x["itemset"]) for x in gt if isinstance(x, dict) and "itemset" in x}

        if not true_sets:
            rewards.append(1.0 if not pred_sets else 0.0)
            continue

        tp = len(pred_sets & true_sets)
        precision = tp / len(pred_sets) if pred_sets else 0.0
        recall = tp / len(true_sets)
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        rewards.append(f1)
    return rewards


def count_accuracy_reward(completions, ground_truth, **kwargs):
    """Fraction of matched itemsets with correct count values."""
    rewards = []
    for text, gt_str in zip(completions, ground_truth):
        predicted = _extract_json(text)
        try:
            gt = json.loads(gt_str)
        except (json.JSONDecodeError, TypeError):
            rewards.append(0.0)
            continue

        if predicted is None:
            rewards.append(0.0)
            continue

        gt_map = {}
        for x in gt:
            if isinstance(x, dict) and "itemset" in x:
                gt_map[frozenset(x["itemset"])] = x.get("count", 0)

        correct = total = 0
        for p in predicted:
            if not isinstance(p, dict):
                continue
            key = frozenset(p.get("itemset", []))
            if key in gt_map:
                total += 1
                if p.get("count") == gt_map[key]:
                    correct += 1

        rewards.append(correct / total if total > 0 else 0.0)
    return rewards


def thinking_reward(completions, **kwargs):
    """Reward for structured reasoning in <think> tags."""
    rewards = []
    for text in completions:
        if "<think>" in text and "</think>" in text:
            think = text.split("<think>", 1)[1].split("</think>", 1)[0]
            # Reward scales with thinking quality
            score = min(1.0, len(think) / 300)
            # Bonus for structured analysis markers
            if any(marker in think.lower() for marker in ["→", "count", "singles", "pairs", "✓"]):
                score = min(1.0, score + 0.2)
            rewards.append(score)
        else:
            rewards.append(0.0)
    return rewards


print("✅ GRPO reward functions defined:")
print("   1. json_format_reward     — Valid JSON with itemset/count/rows schema")
print("   2. itemset_f1_reward      — F1 vs Apriori ground truth")
print("   3. count_accuracy_reward  — Correct counts for matched itemsets")
print("   4. thinking_reward        — Structured <think> reasoning")

In [ ]:
# ── Cell 16: Load DPO checkpoint + GRPO training ─────────────────────────────
from trl import GRPOTrainer, GRPOConfig

# Reload from DPO checkpoint
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = CONFIG["dpo_output_dir"],
    max_seq_length = CONFIG["max_seq_length"],
    load_in_4bit   = CONFIG["load_in_4bit"],
    dtype          = None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                         = CONFIG["lora_r"],
    lora_alpha                = CONFIG["lora_alpha"],
    target_modules            = CONFIG["lora_target_modules"],
    lora_dropout              = 0,
    bias                      = "none",
    use_gradient_checkpointing = "unsloth",
    random_state              = 42,
)

# Format GRPO dataset: apply chat template to prompt
def format_grpo(examples):
    prompts = []
    for prompt_msgs in examples["prompt"]:
        prompts.append(
            tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
        )
    return {"prompt": prompts, "ground_truth": examples["ground_truth"]}

grpo_train = grpo_dataset["train"].map(
    format_grpo, batched=True, remove_columns=grpo_dataset["train"].column_names
)

grpo_trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = [
        json_format_reward,
        itemset_f1_reward,
        count_accuracy_reward,
        thinking_reward,
    ],
    args             = GRPOConfig(
        max_steps                   = CONFIG["grpo_max_steps"],
        per_device_train_batch_size = CONFIG["grpo_batch_size"],
        gradient_accumulation_steps = CONFIG["grpo_grad_accum"],
        learning_rate               = CONFIG["grpo_lr"],
        num_generations             = CONFIG["grpo_num_generations"],
        max_completion_length       = CONFIG["grpo_max_completion_length"],
        warmup_ratio                = 0.05,
        optim                       = "adamw_8bit",
        bf16                        = True,
        fp16                        = False,
        logging_steps               = 5,
        save_steps                  = 50,
        output_dir                  = CONFIG["grpo_output_dir"],
        report_to                   = "none",
        seed                        = 42,
    ),
    train_dataset    = grpo_train,
)

print(f"🔬 Starting GRPO training for {CONFIG['grpo_max_steps']} steps...")
print(f"   Reward functions: json_format, itemset_f1, count_accuracy, thinking")
print(f"   Generations per prompt: {CONFIG['grpo_num_generations']}")
grpo_result = grpo_trainer.train()
print(f"✅ GRPO done!")

In [ ]:
# ── Cell 17: Save GRPO model ─────────────────────────────────────────────────
model.save_pretrained_merged(
    CONFIG["grpo_output_dir"] + "/final",
    tokenizer,
    save_method = "merged_4bit_forced",
)
print(f"💾 GRPO final model saved → {CONFIG['grpo_output_dir']}/final")

---
## Inference Test 🧪

Quick sanity check on a sample CSV to verify the model produces valid JSON with reasoning.

In [ ]:
# ── Cell 19: Quick inference test (v3.16: HF-native inference + correctness check) ──
# v3.14 cleanup: explicit attention masks + cleared max_length on generation
# config to avoid generation warnings during the final sanity check.
#
# v3.15 FIX: tokenizer.apply_chat_template(..., return_tensors="pt") may return
# either a plain tensor or a BatchEncoding depending on transformers/tokenizer
# behavior. Normalize both cases before reading shape / attention mask.
#
# v3.16 FIX: Valid JSON is not enough. Compute ground truth from SAMPLE_CSV and
# compare predicted itemsets/counts/rows so the sanity check catches semantic
# mistakes before Cell 20 pushes anything to the Hub.
import gc, torch, importlib
from itertools import combinations
from transformers import AutoTokenizer, BitsAndBytesConfig

try:
    del model
    gc.collect()
    torch.cuda.empty_cache()
except Exception:
    pass

import transformers.models.qwen2.modeling_qwen2 as _qwen2_mod
_qwen2_mod = importlib.reload(_qwen2_mod)
_Qwen2ForCausalLM = _qwen2_mod.Qwen2ForCausalLM

import peft.peft_model as _peft_mod
_peft_mod = importlib.reload(_peft_mod)
_PeftModel = _peft_mod.PeftModel

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("📦 Loading base model (HF native + SDPA)...")
model = _Qwen2ForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct",
    quantization_config=bnb_config,
    attn_implementation="sdpa",
    dtype=torch.bfloat16,
    device_map={"": 0},
)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
if tokenizer.bos_token_id is None and getattr(model.config, "bos_token_id", None) is not None:
    tokenizer.bos_token_id = model.config.bos_token_id

model = _PeftModel.from_pretrained(model, CONFIG["dpo_output_dir"])
model.eval()
if hasattr(model, "generation_config") and hasattr(model.generation_config, "max_length"):
    model.generation_config.max_length = None
print("✅ Model loaded with final DPO adapter (HF native, SDPA, no Unsloth patches)")

SYSTEM_PROMPT = (
    "You are a frequent itemset extractor. Given CSV transaction data and a "
    "minimum support count, identify all itemsets whose items co-occur in at "
    "least that many rows.\n\n"
    "Rules:\n"
    "1. Scan single items, pairs, and triples (up to size 3)\n"
    "2. Count = number of distinct rows containing ALL items in the itemset\n"
    "3. Only report itemsets with count >= min_support\n"
    "4. Canonicalize items: lowercase, trimmed, sorted alphabetically\n"
    '5. Row references: \"Row N\" format, 1-based indexing\n\n'
    "Think step by step inside <think> tags, then output ONLY a JSON array:\n"
    '[{\"itemset\": [\"item1\", \"item2\"], \"count\": N, \"rows\": [\"Row 1\", \"Row 3\"]}]'
 )

SAMPLE_CSV = """Row 1: bread, milk, eggs
Row 2: bread, butter, jam
Row 3: milk, eggs, cheese
Row 4: bread, milk, eggs, butter
Row 5: bread, eggs"""
MIN_SUPPORT = 2

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f"Find all frequent itemsets with minimum support count = {MIN_SUPPORT} in this dataset:\n\n{SAMPLE_CSV}"},
]

chat_inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
)

if isinstance(chat_inputs, torch.Tensor):
    input_ids = chat_inputs.to("cuda")
    attention_mask = torch.ones_like(input_ids, device=input_ids.device)
else:
    chat_inputs = chat_inputs.to("cuda")
    input_ids = chat_inputs["input_ids"]
    attention_mask = chat_inputs.get("attention_mask")
    if attention_mask is None:
        attention_mask = torch.ones_like(input_ids, device=input_ids.device)

prompt_token_count = input_ids.shape[1]
available_completion_budget = max(1024, CONFIG["max_seq_length"] - prompt_token_count - 64)
think_budget = max(768, min(2000, int(available_completion_budget * 0.55)))
json_budget = max(512, available_completion_budget - think_budget)

response = generate_two_phase(
    model,
    tokenizer,
    input_ids,
    attention_mask   = attention_mask,
    think_temperature = 0.3,
    json_temperature  = 0.05,
    think_max_tokens  = think_budget,
    json_max_tokens   = json_budget,
    top_k             = 50,
    top_p             = 0.90,
 )

print("─── Model Output ───")
print(response)


def parse_sample_transactions(sample_csv: str):
    transactions = []
    for raw_line in sample_csv.strip().splitlines():
        line = raw_line.strip()
        if not line or ":" not in line:
            continue
        row_label, items_part = line.split(":", 1)
        row_label = row_label.strip()
        items = [item.strip().lower() for item in items_part.split(",") if item.strip()]
        transactions.append((row_label, sorted(set(items))))
    return transactions


def build_ground_truth(transactions, min_support: int, max_size: int = 3):
    truth = {}
    for size in range(1, max_size + 1):
        combo_rows = {}
        for row_label, items in transactions:
            for combo in combinations(items, size):
                combo_rows.setdefault(tuple(sorted(combo)), []).append(row_label)
        for combo, rows in combo_rows.items():
            if len(rows) >= min_support:
                truth[combo] = {"count": len(rows), "rows": rows}
    return truth


def normalize_prediction_map(parsed_items):
    prediction_map = {}
    for item in parsed_items:
        if not isinstance(item, dict):
            continue
        raw_itemset = item.get("itemset")
        if not isinstance(raw_itemset, list) or not raw_itemset:
            continue
        key = tuple(sorted(str(token).strip().lower() for token in raw_itemset))
        rows = item.get("rows")
        if not isinstance(rows, list):
            rows = []
        prediction_map[key] = {
            "count": item.get("count"),
            "rows": [str(row).strip() for row in rows],
        }
    return prediction_map


parsed, parse_ok, parsed_json_text = extract_and_repair_json(response)
print(f"\n📋 Has <think> tags: {'<think>' in response and '</think>' in response}")
print(f"📋 Prompt length: {prompt_token_count} tokens")
print(f"📋 Think budget: {think_budget} tokens")
print(f"📋 JSON budget: {json_budget} tokens")

if parse_ok:
    print(f"✅ Valid JSON — {len(parsed)} itemsets found")
    for item in parsed[:5]:
        print(f"  {item.get('itemset')}  count={item.get('count')}  rows={item.get('rows')}")
    if len(parsed) > 5:
        print(f"  ... and {len(parsed) - 5} more")

    transactions = parse_sample_transactions(SAMPLE_CSV)
    expected_map = build_ground_truth(transactions, MIN_SUPPORT, max_size=3)
    predicted_map = normalize_prediction_map(parsed)

    expected_keys = set(expected_map)
    predicted_keys = set(predicted_map)
    missing = sorted(expected_keys - predicted_keys)
    extra = sorted(predicted_keys - expected_keys)
    wrong_counts = []
    wrong_rows = []

    for key in sorted(expected_keys & predicted_keys):
        expected = expected_map[key]
        predicted = predicted_map[key]
        if predicted["count"] != expected["count"]:
            wrong_counts.append((key, predicted["count"], expected["count"]))
        if predicted["rows"] != expected["rows"]:
            wrong_rows.append((key, predicted["rows"], expected["rows"]))

    exact_match = not missing and not extra and not wrong_counts and not wrong_rows
    itemset_precision = len(expected_keys & predicted_keys) / len(predicted_keys) if predicted_keys else 0.0
    itemset_recall = len(expected_keys & predicted_keys) / len(expected_keys) if expected_keys else 0.0
    itemset_f1 = (
        2 * itemset_precision * itemset_recall / (itemset_precision + itemset_recall)
        if (itemset_precision + itemset_recall) > 0
        else 0.0
    )

    print("\n📊 Correctness check against exact ground truth:")
    print(f"   Expected itemsets: {len(expected_keys)}")
    print(f"   Predicted itemsets: {len(predicted_keys)}")
    print(f"   Itemset precision: {itemset_precision:.3f}")
    print(f"   Itemset recall: {itemset_recall:.3f}")
    print(f"   Itemset F1: {itemset_f1:.3f}")
    print(f"   Exact match: {'✅ yes' if exact_match else '❌ no'}")

    if missing:
        print("   Missing itemsets:")
        for key in missing[:5]:
            print(f"     - {list(key)}")
        if len(missing) > 5:
            print(f"     ... and {len(missing) - 5} more")

    if extra:
        print("   Extra itemsets:")
        for key in extra[:5]:
            print(f"     - {list(key)}")
        if len(extra) > 5:
            print(f"     ... and {len(extra) - 5} more")

    if wrong_counts:
        print("   Wrong counts:")
        for key, predicted_count, expected_count in wrong_counts[:5]:
            print(f"     - {list(key)}: predicted={predicted_count}, expected={expected_count}")
        if len(wrong_counts) > 5:
            print(f"     ... and {len(wrong_counts) - 5} more")

    if wrong_rows:
        print("   Wrong row evidence:")
        for key, predicted_rows, expected_rows in wrong_rows[:5]:
            print(f"     - {list(key)}: predicted={predicted_rows}, expected={expected_rows}")
        if len(wrong_rows) > 5:
            print(f"     ... and {len(wrong_rows) - 5} more")

    if exact_match:
        print("\n✅ Sanity check passed — output is both valid JSON and semantically correct.")
    else:
        print("\n⚠️  Sanity check failed — output is parseable, but itemsets/counts/rows are not fully correct.")
        print("   Do not treat this as push-ready quality without running broader evaluation.")
else:
    print("\n⚠️  JSON parse failed even after guarded two-phase generation")
    print("   Full output saved above for debugging")

In [ ]:
# ── Cell 20: Push final model to HuggingFace Hub ─────────────────────────────
# v3: Push as LoRA adapter (save_method="lora"), NOT merged_4bit_forced.
# Council finding: merged_4bit_forced destroys adapter structure and
# produces irreproducible weights. Adapter push is faster and smaller.
import os

if CONFIG["push_to_hub"]:
    hf_token = CONFIG["hf_token"] or os.environ.get("HF_TOKEN", "")

    print(f"🚀 Pushing DPO model to HF Hub: {CONFIG['hf_model_repo']}")

    # Push adapter weights (small, ~65MB)
    model.push_to_hub(
        CONFIG["hf_model_repo"],
        token = hf_token,
    )
    tokenizer.push_to_hub(
        CONFIG["hf_model_repo"],
        token = hf_token,
    )

    print(f"✅ Adapter pushed to: https://huggingface.co/{CONFIG['hf_model_repo']}")
    print(f"   To load: model = PeftModel.from_pretrained(base_model, '{CONFIG['hf_model_repo']}')")

    # Optional: also push merged 16-bit model (for easier loading, ~14GB)
    # Uncomment if you want a standalone model that doesn't need PeftModel:
    # model.push_to_hub_merged(
    #     CONFIG["hf_model_repo"] + "-merged",
    #     tokenizer,
    #     save_method = "merged_16bit",
    #     token = hf_token,
    # )
    # print(f"   Merged 16-bit also pushed to: {CONFIG['hf_model_repo']}-merged")
else:
    print("ℹ️  push_to_hub=False — model only saved locally")
    print(f"   SFT adapter: {CONFIG['sft_output_dir']}")
    print(f"   DPO adapter: {CONFIG['dpo_output_dir']}")